# Exercise 1: Sections Performance Visualization

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Run benchmarks and capture timing
# Expected format: threads, time_serial, time_parallel, sum, max, stddev

## 1. Manual Data Entry
Run the programs and fill in the data below:

In [ ]:
# Example data - replace with your measurements
data = {
    'threads': [1, 2, 3, 4],
    'serial_time': [0.015, 0.015, 0.015, 0.015],  # Should be constant
    'parallel_time': [0.016, 0.010, 0.008, 0.008]  # Replace with actual
}

df = pd.DataFrame(data)
df['speedup'] = df['serial_time'] / df['parallel_time']
df['efficiency'] = df['speedup'] / df['threads']
df

## 2. Performance Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Execution Time
axes[0].plot(df['threads'], df['serial_time'], 'o-', label='Serial', linewidth=2)
axes[0].plot(df['threads'], df['parallel_time'], 's-', label='Parallel', linewidth=2)
axes[0].set_xlabel('Number of Threads')
axes[0].set_ylabel('Time (s)')
axes[0].set_title('Execution Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Speedup
axes[1].plot(df['threads'], df['speedup'], 'o-', label='Actual Speedup', linewidth=2, color='green')
axes[1].plot(df['threads'], df['threads'], '--', label='Ideal (Linear)', linewidth=2, color='red', alpha=0.5)
axes[1].axhline(y=3, linestyle=':', label='Max (3 sections)', color='orange', linewidth=2)
axes[1].set_xlabel('Number of Threads')
axes[1].set_ylabel('Speedup')
axes[1].set_title('Speedup vs Ideal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Efficiency
axes[2].plot(df['threads'], df['efficiency'] * 100, 'o-', label='Efficiency', linewidth=2, color='purple')
axes[2].axhline(y=100, linestyle='--', label='Ideal (100%)', color='red', alpha=0.5)
axes[2].set_xlabel('Number of Threads')
axes[2].set_ylabel('Efficiency (%)')
axes[2].set_title('Parallel Efficiency')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 110)

plt.tight_layout()
plt.savefig('ex1_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Section Workload Visualization

In [ ]:
# Visualize workload distribution across sections
sections = ['Section 1\n(Sum+Mean)', 'Section 2\n(Max)', 'Section 3\n(StdDev)']
ops = [1e6, 1e6, 1e6]  # All do N operations
dependencies = ['None', 'None', 'Waits for Section 1']

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.barh(sections, ops, color=colors, alpha=0.7, edgecolor='black')

# Add dependency annotations
for i, (section, dep) in enumerate(zip(sections, dependencies)):
    ax.text(ops[i] + 0.05e6, i, dep, va='center', fontsize=10, style='italic')

ax.set_xlabel('Number of Operations (N)', fontsize=12)
ax.set_title('Workload Distribution Across Sections', fontsize=14, weight='bold')
ax.set_xlim(0, 1.5e6)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('ex1_workload.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Key Observations

- **Max useful threads**: 3 (one per section)
- **Expected speedup**: < 3x due to:
  - Busy-waiting overhead in Section 3
  - Thread creation/destruction costs
  - Load imbalance (if any section finishes faster)
- **Better approach**: Use `reduction` clauses instead of `sections`

## 5. Overhead Analysis

In [ ]:
# Calculate overhead
df['overhead'] = df['parallel_time'] - (df['serial_time'] / df['threads'])
df['overhead_pct'] = (df['overhead'] / df['serial_time']) * 100

print("Overhead Analysis:")
print(df[['threads', 'overhead', 'overhead_pct']])

# Plot overhead
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df['threads'], df['overhead_pct'], 'o-', linewidth=2, markersize=8, color='darkred')
ax.set_xlabel('Number of Threads', fontsize=12)
ax.set_ylabel('Overhead (%)', fontsize=12)
ax.set_title('Parallelization Overhead', fontsize=14, weight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ex1_overhead.png', dpi=150, bbox_inches='tight')
plt.show()